In [1]:
# Import necessary libraries
import os
from langchain_community.vectorstores import FAISS
from langchain.vectorstores import FAISS
#from langchain_openai import OpenAIEmbeddings
from langchain.chains.conversational_retrieval.base import ConversationalRetrievalChain
from flask import Flask, render_template, request, redirect
from PyPDF2 import PdfReader
from langchain_openai import ChatOpenAI
from langchain.text_splitter import CharacterTextSplitter
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain.llms.openai import OpenAI 
import glob
from flask import Flask, render_template, request, redirect, url_for
from langdetect import detect
from duodecim_scraper import search_duodecim
import json
from langchain.prompts import PromptTemplate
from langchain.chains.qa_with_sources import load_qa_with_sources_chain

In [2]:
OPENAI_API_KEY = "your-openai-api-key-here"
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY

# Text Generation

In [4]:
# Flask App
app = Flask(__name__)

# Global state
vectorstore = None
conversation_chain = None
chat_history = []
last_user_question = ""

# Load all PDFs from a folder
def load_pdfs_from_folder(folder_path):
    pdf_paths = glob.glob(os.path.join(folder_path, '*.pdf'))
    pdf_files = [open(path, 'rb') for path in pdf_paths]
    return pdf_files

# Load JSON chunks with source metadata
def load_json_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


# Translate English question to Finnish
def translate_to_finnish(question):
    llm = ChatOpenAI(model_name="gpt-4o")
    prompt = f"Translate the following question to Finnish:\n\n{question}"
    return llm.predict(prompt).strip()

# Translate Finnish answer to English
def translate_to_english(answer):
    llm = ChatOpenAI(model_name="gpt-4o")
    prompt = f"Translate the following answer to English:\n\n{answer}"
    return llm.predict(prompt).strip()

# Chat route
@app.route('/chat', methods=['GET', 'POST'])
def chat():
    global vectorstore, conversation_chain, chat_history, last_user_question

    if request.method == 'POST':
        user_question = request.form['user_question']
        last_user_question = user_question
        lang = detect(user_question)

        if len(user_question.strip().split()) < 3:
            last_user_msg = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), "")
            user_question = f"{last_user_msg}\n{user_question}"

        if lang == 'en':
            used_question = translate_to_finnish(user_question)
        else:
            used_question = user_question

        response = conversation_chain({'question': used_question})
        sources = response.get('source_documents', [])
        source_urls = set()

        for doc in sources:
            if "source" in doc.metadata and doc.metadata["source"] != "PDF":
                clean_url = doc.metadata["source"].split("#")[0]  # Remove fragment identifiers
                source_urls.add(clean_url)

        if not sources or all(not doc.page_content.strip() for doc in sources):
            answer = "Tietojeni perusteella en osaa vastata tähän kysymykseen."
        else:
            answer = response.get('answer', "Tietojeni perusteella en osaa vastata tähän kysymykseen.")

        # Translate back if original question was in English
        if lang == 'en': #and "en osaa vastata" not in answer
            answer = translate_to_english(answer)
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Source: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"
        else:
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Lähde: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"

        chat_history.append({'type': 'user', 'content': user_question})
        chat_history.append({'type': 'assistant', 'content': answer})

    return render_template('chat.html', chat_history=chat_history)


@app.route('/search_duodecim', methods=['POST'])
def search_duodecim_route():
    global chat_history

    # Get the last user question
    last_question = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), None)
    if not last_question:
        return redirect(url_for('chat'))

    try:
        from duodecim_scraper import search_duodecim
        result = search_duodecim(last_question, headless=True)
        duodecim_answer = f"<strong>🔍 Duodecim AI:</strong><br>{result['answer']}"

        # Remove duplicate title+url combinations
        unique_links = {}
        for src in result['sources']:
            key = (src['title'], src['url'])
            if key not in unique_links:
                unique_links[key] = f'<a href="{src["url"]}" target="_blank">📎 {src["title"]}: {src["url"]}</a>'

        if unique_links:
            duodecim_answer += "<br><br><strong>SOURCES:</strong><br>" + "<br>".join(unique_links.values())

        chat_history.append({'type': 'assistant', 'content': duodecim_answer})

    except Exception as e:
        chat_history.append({'type': 'assistant', 'content': f"<strong>Duodecim AI:</strong> An error occurred while fetching the answer. {e}"})

    return redirect(url_for('chat'))







# Extract raw text
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        pdf_reader = PdfReader(pdf)
        for page in pdf_reader.pages:
            content = page.extract_text()
            if content:
                text += content
    return text

# Split text into chunks
def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    return text_splitter.split_text(text)

# Build the vectorstore from both PDFs and JSON chunks
def build_vectorstore_from_sources(folder_path):
    embeddings = OpenAIEmbeddings()
    all_chunks = []

    # Add PDF chunks
    pdf_files = load_pdfs_from_folder(folder_path)
    pdf_text = get_pdf_text(pdf_files)
    for chunk in get_text_chunks(pdf_text):
        all_chunks.append({"text": chunk, "metadata": {"source": "PDF"}})

    # Add JSON chunks (from scraper)
    json_path = os.path.join(folder_path, "uniapnea_chunks.json")
    if os.path.exists(json_path):
        json_chunks = load_json_chunks(json_path)
        for item in json_chunks:
            all_chunks.append({"text": item["text"], "metadata": {"source": item["source"]}})

    texts = [x["text"] for x in all_chunks]
    metadatas = [x["metadata"] for x in all_chunks]

    return FAISS.from_texts(texts=texts, embedding=embeddings, metadatas=metadatas)



# Conversation chain builder with friendly tone
def get_conversation_chain(vectorstore):
    llm = ChatOpenAI(model_name="gpt-4o")

    # Friendly, easy-to-understand prompt
    friendly_prompt = PromptTemplate(
        input_variables=["question", "context", "chat_history"],
        template="""
Olet avulias ja selkeä suomenkielinen assistentti. Selität asiat yksinkertaisesti, ystävällisesti ja helposti ymmärrettävällä tavalla – kuin selittäisit isoäidille, joka ei ole tottunut tekniikkaan tai vaikeisiin termeihin.

Muista:
- Käytä lyhyitä ja helppoja lauseita.
- Vältä vaikeita sanoja ja termejä, tai selitä ne helposti.
- Käytä esimerkkejä tai vertauksia, jos ne auttavat ymmärtämään.

Tässä keskustelun aiempi sisältö:
{chat_history}

Tässä käyttäjän kysymys:
{question}

Tässä asiayhteys:
{context}

Anna selkeä, lempeä ja ymmärrettävä vastaus.
"""
    )

    memory = ConversationBufferMemory(
        memory_key='chat_history',
        return_messages=True,
        output_key='answer'
    )

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity_score_threshold",
            search_kwargs={"score_threshold": 0.76}
        ),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": friendly_prompt},
        output_key="answer"
    )












# Home page
@app.route('/')
def index():
    return render_template('index.html')


# Load uniapnea folder
@app.route('/load/uniapnea')
def load_uniapnea_folder():
    global vectorstore, conversation_chain, chat_history
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    vectorstore = build_vectorstore_from_sources(folder_path)
    conversation_chain = get_conversation_chain(vectorstore)
    chat_history = []
    return redirect(url_for('chat'))



# Load folder on startup
if __name__ == '__main__':
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    if os.path.exists(folder_path):
        try:
            vectorstore = build_vectorstore_from_sources(folder_path)
            conversation_chain = get_conversation_chain(vectorstore)
            print("📁 Folder and .json loaded on startup!")
        except Exception as e:
            print(f"Error loading folder: {e}")
    app.run()


Error loading folder: Error code: 400 - {'error': {'message': 'Requested 383545 tokens, max 300000 tokens per request', 'type': 'max_tokens_per_request', 'param': None, 'code': 'max_tokens_per_request'}}
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [23/Jun/2025 13:52:33] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [23/Jun/2025 13:52:34] "GET /static/Background.png HTTP/1.1" 200 -
[2025-06-23 13:53:00,557] ERROR in app: Exception on /load/uniapnea [GET]
Traceback (most recent call last):
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 1473, in wsgi_app
    response = self.full_dispatch_request()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 882, in full_dispatch_request
    rv = self.handle_user_exception(e)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 880, in full_dispatch_request
    rv = self.dispatch_request()
         ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\flask\app.py", line 865, in dispatch_request
  

In [13]:
# Flask App
app = Flask(__name__)

# Global state
vectorstore = None
conversation_chain = None
chat_history = []
last_user_question = ""

# Load all PDFs from a folder
def load_pdfs_from_folder(folder_path):
    pdf_paths = glob.glob(os.path.join(folder_path, '*.pdf'))
    pdf_files = [open(path, 'rb') for path in pdf_paths]
    return pdf_files

# Load JSON chunks with source metadata
def load_json_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Translate English question to Finnish
def translate_to_finnish(question):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")
    prompt = f"Translate the following question to Finnish:\n\n{question}"
    return llm.predict(prompt).strip()

# Translate Finnish answer to English
def translate_to_english(answer):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")
    prompt = f"Translate the following answer to English:\n\n{answer}"
    return llm.predict(prompt).strip()

# Chat route
@app.route('/chat', methods=['GET', 'POST'])
def chat():
    global vectorstore, conversation_chain, chat_history, last_user_question

    if request.method == 'POST':
        user_question = request.form['user_question']
        last_user_question = user_question
        
        # Better language detection with fallbacks - FIXED
        try:
            lang = detect(user_question)
            # Fix common misdetections
            if lang not in ['en', 'fi'] or len(user_question.strip()) < 10:
                # For short phrases, check if it's obviously English or Finnish
                english_words = ['what', 'is', 'how', 'can', 'you', 'explain', 'more', 'about', 'it', 'the', 'this', 'that']
                finnish_words = ['mikä', 'mitä', 'miten', 'voitko', 'selitä', 'lisää', 'tästä', 'kuinka', 'on', 'kerro']
                
                user_lower = user_question.lower()
                english_count = sum(1 for word in english_words if word in user_lower)
                finnish_count = sum(1 for word in finnish_words if word in user_lower)
                
                if finnish_count > english_count:
                    lang = 'fi'
                elif english_count > 0:
                    lang = 'en'
                else:
                    lang = 'en'  # Default to English for unclear cases
        except:
            lang = 'en'  # Default fallback
        
        print(f"Language detection: '{user_question}' -> {lang}")
        
        # Don't override language detection with conversation history - let each question be independent
        # (Removed the conversation language maintenance that was causing problems)

        # Simple context enhancement like your original - only for very short questions
        original_question = user_question  # Store original before any modification
        if len(user_question.strip().split()) < 3:  # Back to < 3 like your original
            last_user_msg = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), "")
            user_question = f"{last_user_msg}\n{user_question}"
            print(f"Enhanced question: {user_question[:100]}...")

        # Translate to Finnish for processing, but remember original language
        original_lang = lang
        if lang == 'en':
            used_question = translate_to_finnish(user_question)
        else:
            used_question = user_question

        response = conversation_chain({'question': used_question})
        sources = response.get('source_documents', [])
        source_urls = set()

        # DEBUG: Print what sources were found
        print(f"\n=== DEBUG INFO ===")
        print(f"Original question: {original_question}")
        print(f"Detected language: {original_lang}")
        print(f"Used question: {used_question}")
        print(f"Found {len(sources)} source documents")
        for i, doc in enumerate(sources[:3]):  # Show first 3 sources
            print(f"Source {i+1}: {doc.page_content[:200].replace(chr(10), ' ')}")
            print(f"Metadata: {doc.metadata}")
            print("---")
        print(f"Raw answer: {response.get('answer', 'No answer')[:300]}...")
        print(f"==================\n")

        # Check if we have relevant sources and good content
        if not sources or all(not doc.page_content.strip() for doc in sources):
            answer = "Tietojeni perusteella en osaa vastata tähän kysymykseen."
        else:
            answer = response.get('answer', "Tietojeni perusteella en osaa vastata tähän kysymykseen.")
            
            # Additional check: if answer seems to be refusing despite having sources
            if ("en osaa vastata" in answer or "cannot answer" in answer) and len(sources) > 0:
                # Check if any source actually contains relevant content
                relevant_content = []
                search_terms = used_question.lower().split()
                for doc in sources:
                    content_lower = doc.page_content.lower()
                    if any(term in content_lower for term in search_terms if len(term) > 3):
                        relevant_content.append(doc.page_content[:200])
                
                if relevant_content:
                    print(f"Found relevant content but AI refused: {relevant_content[0][:100]}...")

        for doc in sources:
            if "source" in doc.metadata and doc.metadata["source"] != "PDF":
                clean_url = doc.metadata["source"].split("#")[0]
                source_urls.add(clean_url)

        if lang == 'en':
            answer = translate_to_english(answer)
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Source: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"
        else:
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Lähde: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"

        chat_history.append({'type': 'user', 'content': original_question})
        chat_history.append({'type': 'assistant', 'content': answer})

    return render_template('chat.html', chat_history=chat_history)

@app.route('/search_duodecim', methods=['POST'])
def search_duodecim_route():
    global chat_history

    last_question = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), None)
    if not last_question:
        return redirect(url_for('chat'))

    try:
        from duodecim_scraper import search_duodecim
        result = search_duodecim(last_question, headless=True)
        duodecim_answer = f"<strong>🔍 Duodecim AI:</strong><br>{result['answer']}"

        unique_links = {}
        for src in result['sources']:
            key = (src['title'], src['url'])
            if key not in unique_links:
                unique_links[key] = f'<a href="{src["url"]}" target="_blank">📎 {src["title"]}: {src["url"]}</a>'

        if unique_links:
            duodecim_answer += "<br><br><strong>SOURCES:</strong><br>" + "<br>".join(unique_links.values())

        chat_history.append({'type': 'assistant', 'content': duodecim_answer})

    except Exception as e:
        chat_history.append({'type': 'assistant', 'content': f"<strong>Duodecim AI:</strong> An error occurred while fetching the answer. {e}"})

    return redirect(url_for('chat'))

# Extract raw text
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        pdf_reader = PdfReader(pdf)
        for page in pdf_reader.pages:
            content = page.extract_text()
            if content:
                text += content
    return text

# Split text into smaller chunks - REDUCED CHUNK SIZE
def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500,  # Reduced from 1000
        chunk_overlap=100,  # Reduced from 200
        length_function=len
    )
    return text_splitter.split_text(text)

# NEW: Process embeddings in batches to avoid token limits
def create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=50):
    """Process embeddings in smaller batches to avoid token limits"""
    vectorstores = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_metadatas = metadatas[i:i + batch_size]
        batch_num = i // batch_size + 1
        
        # Progress indicator every 25 batches to reduce spam
        if batch_num % 25 == 0 or batch_num == 1 or batch_num == total_batches:
            print(f"Processing batch {batch_num}/{total_batches} ({batch_num/total_batches*100:.1f}%)")
        
        try:
            # Create vectorstore for this batch
            batch_vectorstore = FAISS.from_texts(
                texts=batch_texts, 
                embedding=embeddings, 
                metadatas=batch_metadatas
            )
            vectorstores.append(batch_vectorstore)
            
        except Exception as e:
            print(f"Error processing batch {batch_num}: {e}")
            # Skip this batch and continue
            continue
    
    print(f"✅ Successfully processed {len(vectorstores)}/{total_batches} batches")
    
    # Merge all vectorstores
    if vectorstores:
        print("🔗 Merging vectorstores...")
        main_vectorstore = vectorstores[0]
        for vs in vectorstores[1:]:
            main_vectorstore.merge_from(vs)
        return main_vectorstore
    else:
        raise Exception("No vectorstores were created successfully")

# Build the vectorstore from both PDFs and JSON chunks - UPDATED WITH BATCH PROCESSING
def build_vectorstore_from_sources(folder_path):
    embeddings = OpenAIEmbeddings()
    all_chunks = []

    # Add PDF chunks
    pdf_files = load_pdfs_from_folder(folder_path)
    if pdf_files:
        print("Processing PDF files...")
        pdf_text = get_pdf_text(pdf_files)
        print(f"Total PDF text length: {len(pdf_text)} characters")
        
        pdf_chunks = get_text_chunks(pdf_text)
        print(f"Created {len(pdf_chunks)} PDF chunks")
        
        for chunk in pdf_chunks:
            all_chunks.append({"text": chunk, "metadata": {"source": "PDF"}})

    # Add JSON chunks (from scraper)
    json_path = os.path.join(folder_path, "uniapnea_chunks.json")
    if os.path.exists(json_path):
        print("Processing JSON chunks...")
        json_chunks = load_json_chunks(json_path)
        print(f"Loaded {len(json_chunks)} JSON chunks")
        
        for item in json_chunks:
            all_chunks.append({"text": item["text"], "metadata": {"source": item["source"]}})

    print(f"Total chunks to process: {len(all_chunks)}")
    
    if not all_chunks:
        raise Exception("No content found to process")

    texts = [x["text"] for x in all_chunks]
    metadatas = [x["metadata"] for x in all_chunks]

    # Use batch processing instead of processing all at once (larger batches = faster)
    return create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=100)

# Conversation chain builder with friendly tone
def get_conversation_chain(vectorstore):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")

    friendly_prompt = PromptTemplate(
        input_variables=["question", "context", "chat_history"],
        template="""
Olet avulias assistentti. Vastaat vain annettujen lähdemateriaalien perusteella.

SÄÄNNÖT:
- Jos käyttäjä kysyy vertailua (esim. "difference between", "compare", "vs"), vastaa vertaillen
- Jos kysytään vain yhdestä aiheesta, kerro vain siitä - älä mainitse muita aiheita
- Jos asiayhteys sisältää tietoa kysytystä aiheesta/aiheista, vastaa sen perusteella  
- Jos kysymys on englanniksi, vastaa englanniksi
- Jos kysymys on suomeksi, vastaa suomeksi
- Jos asiayhteys ei sisällä tietoa kysytystä aiheesta, sano: "Tietojeni perusteella en osaa vastata tähän kysymykseen."

Tässä keskustelun aiempi sisältö:
{chat_history}

Tässä käyttäjän kysymys:
{question}

Tässä asiayhteys:
{context}

Vastaus:
"""
    )

    memory = ConversationBufferMemory(
        memory_key='chat_history',
        return_messages=True,
        output_key='answer'
    )

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity", 
            search_kwargs={"k": 15}  # Get more chunks, no threshold restriction
        ),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": friendly_prompt},
        output_key="answer"
    )

        # Clear chat button route
@app.route('/clear_chat')
def clear_chat():
    global chat_history
    chat_history = []
    return redirect(url_for('chat'))

# Home page
@app.route('/')
def index():
    return render_template('index.html')
@app.route('/load/uniapnea')
def load_uniapnea_folder():
    global vectorstore, conversation_chain, chat_history
    
    # Check if already loaded
    if vectorstore is not None:
        print("⚠️ Vectorstore already loaded, skipping reload")
        chat_history = []  # Just clear chat history
        return redirect(url_for('chat'))
    
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    
    try:
        print("Starting to load vectorstore...")
        vectorstore = build_vectorstore_from_sources(folder_path)
        conversation_chain = get_conversation_chain(vectorstore)
        chat_history = []
        print("✅ Successfully loaded vectorstore!")
        return redirect(url_for('chat'))
        
    except Exception as e:
        print(f"❌ Error loading vectorstore: {e}")
        return f"<h1>Error Loading Data</h1><p>Error: {str(e)}</p><a href='/'>Go back to home</a>", 500

# Load folder on startup - UPDATED WITH BETTER ERROR HANDLING
if __name__ == '__main__':
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    if os.path.exists(folder_path):
        try:
            print("Loading vectorstore on startup...")
            vectorstore = build_vectorstore_from_sources(folder_path)
            conversation_chain = get_conversation_chain(vectorstore)
            print("📁 Folder and .json loaded on startup!")
        except Exception as e:
            print(f"⚠️ Error loading folder on startup: {e}")
            print("The app will start without pre-loaded data. Use /load/uniapnea to load data manually.")
    else:
        print(f"⚠️ Folder path does not exist: {folder_path}")
        
    app.run(debug=False)  # Set to False to avoid restart loop in production

Loading vectorstore on startup...
Processing PDF files...
Total PDF text length: 2683152 characters
Created 6831 PDF chunks
Processing JSON chunks...
Loaded 10 JSON chunks
Total chunks to process: 6841
Processing batch 1/69 (1.4%)
Processing batch 25/69 (36.2%)
Processing batch 50/69 (72.5%)
Processing batch 69/69 (100.0%)
✅ Successfully processed 69/69 batches
🔗 Merging vectorstores...
📁 Folder and .json loaded on startup!
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [26/Jun/2025 09:19:50] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [26/Jun/2025 09:19:51] "GET /static/Background.png HTTP/1.1" 304 -
127.0.0.1 - - [26/Jun/2025 09:20:01] "GET /load/uniapnea HTTP/1.1" 302 -
127.0.0.1 - - [26/Jun/2025 09:20:01] "GET /chat HTTP/1.1" 200 -


⚠️ Vectorstore already loaded, skipping reload


127.0.0.1 - - [26/Jun/2025 09:39:37] "GET /chat/chat HTTP/1.1" 404 -
127.0.0.1 - - [26/Jun/2025 09:40:12] "GET /chat HTTP/1.1" 200 -


Language detection: 'what is sleep apnea?' -> en

=== DEBUG INFO ===
Original question: what is sleep apnea?
Detected language: en
Used question: Mitä on uniapnea?
Found 15 source documents
Source 1: | Kelan etuudet | Toimintakyvyn rajoittuessa | Päiväraha, kuntoutusraha, tuki |   | Vertaistuki | Kaikille | Ryhmiä, tapahtumia,  tukea |    UNIAPNEAN YMMÄRTÄMINEN   Mikä uniapnea on?    Uniapnea on u
Metadata: {'source': 'PDF'}
---
Source 2: Mieltymykset Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optimoimiseksi. Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optim
Metadata: {'source': 'https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea'}
---
Source 3: Mieltymykset Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optimoimiseksi. Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optim
Metadata: {'source': 'https://www.terveyskyl

127.0.0.1 - - [26/Jun/2025 09:40:28] "POST /chat HTTP/1.1" 200 -



=== DEBUG INFO ===
Original question: what is sleep apnea?
Detected language: en
Used question: Mikä on uniapnea?
Found 15 source documents
Source 1: | Kelan etuudet | Toimintakyvyn rajoittuessa | Päiväraha, kuntoutusraha, tuki |   | Vertaistuki | Kaikille | Ryhmiä, tapahtumia,  tukea |    UNIAPNEAN YMMÄRTÄMINEN   Mikä uniapnea on?    Uniapnea on u
Metadata: {'source': 'PDF'}
---
Source 2: Mieltymykset Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optimoimiseksi. Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optim
Metadata: {'source': 'https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#ch2-settings'}
---
Source 3: Mieltymykset Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optimoimiseksi. Käytetään liikenteen jakamiseen verkkosivustolle useille palvelimille vasteaikojen optim
Metadata: {'source': 'https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauk

127.0.0.1 - - [26/Jun/2025 09:40:32] "POST /chat HTTP/1.1" 200 -


Language detection: 'what is copd?' -> en

=== DEBUG INFO ===
Original question: what is copd?
Detected language: en
Used question: Mikä on COPD?
Found 15 source documents
Source 1: ja emfyseeman (keuhkolaajentuman).   COPD (Chronic Obstructive Pulmonary Disease) on kansainvälinen kattotermi samalle  sairaudelle.   Emfyseema tarkoittaa keuhkolaajentumaa, jossa keuhkorakkulat vaur
Metadata: {'source': 'PDF'}
---
Source 2: Pyydä hoitajaltasi tai apteekista ohjeet inhalaattorin oikeaan käyttöön.   Kuinka usein minun tulee tehdä PEF -mittauksia?  Säännöllisesti oireiden seuraamiseksi ja aina, jos tunnet olosi huonontuvan.
Metadata: {'source': 'PDF'}
---
Source 3: Ikääntynee t: COPD:n ja astman erottaminen on tärkeää, sillä niiden hoito eroaa toisistaan.  Lisäksi muut hengityselinten sairaudet ovat yleisempiä tässä ikäryhmässä.   3.1 Miksi astman diagnoosi on t
Metadata: {'source': 'PDF'}
---
Raw answer: Tietojeni perusteella COPD (Chronic Obstructive Pulmonary Disease) on pitkäaikainen keu

127.0.0.1 - - [26/Jun/2025 09:40:59] "POST /chat HTTP/1.1" 200 -


In [3]:
import os
import json
import glob
from flask import Flask, render_template, request, redirect, url_for, jsonify
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain import PromptTemplate
from langdetect import detect
import traceback

# Flask App
app = Flask(__name__)

# Add these configurations
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024 # 16MB max request size
app.config['WTF_CSRF_ENABLED'] = False # Disable CSRF for now (consider enabling for production)

# Global state variables for the application
vectorstore = None
conversation_chain = None
chat_history = [] # Stores a list of dictionaries: [{'type': 'user', 'content': '...'}, {'type': 'assistant', 'content': '...'}]

# --- PDF Loading and Processing Functions ---

def load_pdfs_from_folder(folder_path):
    """Loads all PDF files from a specified folder."""
    pdf_paths = glob.glob(os.path.join(folder_path, '*.pdf'))
    # Open files in binary read mode ('rb')
    pdf_files = [open(path, 'rb') for path in pdf_paths]
    print(f"Loaded {len(pdf_files)} PDF files from {folder_path}")
    return pdf_files

def load_json_chunks(file_path):
    """Loads JSON data (e.g., scraped content chunks) from a file."""
    if not os.path.exists(file_path):
        print(f"JSON file not found: {file_path}")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        print(f"Loaded {len(data)} JSON chunks from {file_path}")
        return data

def get_pdf_text(pdf_docs):
    """Extracts raw text from a list of PDF documents."""
    text = ""
    for pdf in pdf_docs:
        try:
            pdf_reader = PdfReader(pdf)
            for page in pdf_reader.pages:
                content = page.extract_text()
                if content:
                    text += content
        except Exception as e:
            print(f"Error reading PDF {pdf.name}: {e}")
            continue
    return text

def get_text_chunks(text):
    """Splits a long string of text into smaller, overlapping chunks."""
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500, # Max characters per chunk
        chunk_overlap=100, # Overlap to maintain context between chunks
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

def create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=100):
    """
    Processes embeddings in smaller batches to avoid OpenAI token limits
    and creates a FAISS vectorstore. Merges individual batch vectorstores
    into a single main vectorstore.
    """
    vectorstores = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    
    print(f"Starting to create embeddings in {total_batches} batches...")
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_metadatas = metadatas[i:i + batch_size]
        batch_num = i // batch_size + 1
        
        print(f"Processing batch {batch_num}/{total_batches} ({batch_num/total_batches*100:.1f}%) with {len(batch_texts)} chunks...")
        
        try:
            batch_vectorstore = FAISS.from_texts(
                texts=batch_texts, 
                embedding=embeddings, 
                metadatas=batch_metadatas
            )
            vectorstores.append(batch_vectorstore)
            
        except Exception as e:
            print(f"Error processing batch {batch_num}: {e}")
            continue
    
    print(f"✅ Successfully processed {len(vectorstores)}/{total_batches} batches.")
    
    if vectorstores:
        print("🔗 Merging vectorstores...")
        main_vectorstore = vectorstores[0]
        for vs in vectorstores[1:]:
            main_vectorstore.merge_from(vs)
        print("✅ Merged all vectorstores.")
        return main_vectorstore
    else:
        raise Exception("No vectorstores were created successfully. Check input data and API key.")

def build_vectorstore_from_sources(folder_path):
    """
    Builds a FAISS vectorstore from both PDF documents and JSON chunks.
    Embeddings are created using OpenAIEmbeddings.
    """
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    all_chunks_data = [] # List of {'text': '...', 'metadata': {...}}

    # --- Process PDFs ---
    pdf_files = load_pdfs_from_folder(folder_path)
    if pdf_files:
        print("Extracting text from PDF files...")
        pdf_text = get_pdf_text(pdf_files)
        print(f"Total extracted PDF text length: {len(pdf_text)} characters")
        
        pdf_chunks = get_text_chunks(pdf_text)
        print(f"Created {len(pdf_chunks)} PDF chunks.")
        
        for chunk in pdf_chunks:
            all_chunks_data.append({"text": chunk, "metadata": {"source": "PDF"}})

    # --- Process JSON Chunks (from web scraper, etc.) ---
    json_path = os.path.join(folder_path, "uniapnea_chunks.json") # Assumed file name
    if os.path.exists(json_path):
        print(f"Loading JSON chunks from {json_path}...")
        json_chunks = load_json_chunks(json_path)
        
        for item in json_chunks:
            # Ensure 'source' is always present in metadata
            source_url = item.get("source", "Unknown") # Use .get() for safety
            all_chunks_data.append({"text": item["text"], "metadata": {"source": source_url}})
    else:
        print(f"JSON chunk file not found at {json_path}. Skipping JSON loading.")

    print(f"Total chunks collected from all sources: {len(all_chunks_data)}")
    
    if not all_chunks_data:
        raise Exception("No content found to process for vectorstore creation. Check PDF folder and JSON file path.")

    # Separate texts and metadatas for embedding creation
    texts = [x["text"] for x in all_chunks_data]
    metadatas = [x["metadata"] for x in all_chunks_data]

    # Create and return the merged vectorstore
    return create_embeddings_in_batches(texts, metadatas, embeddings)

# --- LLM and Conversation Chain Setup ---

def get_conversation_chain(vectorstore):
    """
    Initializes and returns a ConversationalRetrievalChain for the chatbot.
    Configured with a specific prompt template for friendly and contextual answers.
    """
    llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7) # Added temperature for creativity

    friendly_prompt = PromptTemplate(
        input_variables=["question", "context", "chat_history"],
        template="""
You are a helpful assistant. You only answer based on the provided source materials.

RULES:
- If the user asks for a comparison (e.g., "difference between", "compare", "vs"), answer by comparing them.
- If only one topic is asked, only talk about that - do not mention other topics.
- If the context contains information about the requested topic(s), answer based on it.
- If the question is in English, answer in English.
- If the question is in Finnish, answer in Finnish.
- If the context does not contain information about the requested topic, say: "Based on my knowledge, I cannot answer this question." (or the Finnish equivalent).

Here is the previous conversation history:
{chat_history}

Here is the user's question:
{question}

Here is the context relevant to the question:
{context}

Answer:
"""
    )

    # Memory for maintaining conversation context
    memory = ConversationBufferMemory(
        memory_key='chat_history', # Key to store chat history in the chain's memory
        return_messages=True, # Return messages as objects (rather than just strings)
        output_key='answer' # The key where the final answer will be stored in the chain's output
    )

    # ConversationalRetrievalChain setup
    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity", # Uses cosine similarity for retrieval
            search_kwargs={"k": 15} # Number of relevant document chunks to retrieve
        ),
        memory=memory, # Pass the configured memory
        return_source_documents=True, # Return the source documents used for the answer
        combine_docs_chain_kwargs={"prompt": friendly_prompt}, # Use the custom prompt
        output_key="answer" # Ensure the output key is consistent
    )

# --- Language Translation Functions ---

def translate_to_finnish(question):
    """Translates an English question to Finnish using an LLM."""
    try:
        llm = ChatOpenAI(model_name="gpt-4o-mini")
        prompt = f"Translate the following question to Finnish:\n\n{question}"
        return llm.predict(prompt).strip()
    except Exception as e:
        print(f"Translation error (to Finnish): {e}")
        return question # Return original if translation fails

def translate_to_english(answer):
    """Translates a Finnish answer to English using an LLM."""
    try:
        llm = ChatOpenAI(model_name="gpt-4o-mini")
        prompt = f"Translate the following answer to English:\n\n{answer}"
        return llm.predict(prompt).strip()
    except Exception as e:
        print(f"Translation error (to English): {e}")
        return answer # Return original if translation fails

# --- Flask Routes ---

@app.route('/')
def index():
    """Renders the home page."""
    return render_template('index.html')

@app.route('/chat', methods=['GET'])
def chat():
    """
    Renders the main chat interface.
    This route now only handles GET requests and displays the current chat history.
    All question processing is handled via AJAX to /process_question.
    """
    return render_template('chat.html', chat_history=chat_history)

@app.route('/process_question', methods=['POST'])
def process_question():
    """
    Receives a user question via AJAX, processes it with the LLM,
    and returns the answer and sources. Updates server-side chat_history.
    """
    global vectorstore, conversation_chain, chat_history # Access global state

    try:
        data = request.get_json()
        if not data or 'question' not in data:
            return jsonify({'error': 'No question provided'}), 400
            
        user_question = data['question'].strip()
        if not user_question:
            return jsonify({'error': 'Empty question'}), 400
            
        print(f"\n=== Processing question: {user_question} ===")
        
        # Check if conversation chain is ready (system initialized)
        if conversation_chain is None:
            return jsonify({'error': 'System not initialized. Please load data first.'}), 500
        
        # Language detection for input question
        try:
            lang = detect(user_question)
            # Fallback for very short inputs or unreliable detection
            if lang not in ['en', 'fi'] or len(user_question.strip()) < 10:
                english_words = ['what', 'is', 'how', 'can', 'you', 'explain', 'more', 'about', 'it', 'the', 'this', 'that']
                finnish_words = ['mikä', 'mitä', 'miten', 'voitko', 'selitä', 'lisää', 'tästä', 'kuinka', 'on', 'kerro']
                
                user_lower = user_question.lower()
                english_count = sum(1 for word in english_words if word in user_lower)
                finnish_count = sum(1 for word in finnish_words if word in user_lower)
                
                lang = 'fi' if finnish_count > english_count else 'en'
        except:
            lang = 'en' # Default to English if detection fails
            
        print(f"Detected language: {lang}")
        
        # Translate question to Finnish if original is English, as the prompt is primarily Finnish-centric
        original_lang = lang
        if lang == 'en':
            used_question = translate_to_finnish(user_question)
            print(f"Translated to Finnish for processing: {used_question}")
        else:
            used_question = user_question # Use original if already Finnish
            
        # Process with conversation chain. The `chat_history` for the chain is managed by `memory`.
        # We pass the `used_question` (possibly translated).
        print("Querying conversation chain...")
        response = conversation_chain({'question': used_question})
        
        # Extract answer and source documents
        answer = response.get('answer', 'Tietojeni perusteella en osaa vastata tähän kysymykseen.')
        sources = response.get('source_documents', [])
        source_urls = set()
        
        print(f"Received {len(sources)} sources.")
        print(f"Raw answer preview: {answer[:200]}...") # Log a preview
        
        # Collect unique source URLs from metadata
        for doc in sources:
            if "source" in doc.metadata and doc.metadata["source"] != "PDF": # Exclude generic PDF source
                clean_url = doc.metadata["source"].split("#")[0] # Remove fragment identifiers
                source_urls.add(clean_url)
                
        # Translate the answer back to English if the original question was in English
        if original_lang == 'en' and answer:
            answer = translate_to_english(answer)
            print(f"Translated answer back to English: {answer[:200]}...")
            
        # Append source links to the answer for display
        if source_urls:
            if original_lang == 'en':
                joined_sources = "<br>".join(f'<a href="{url}" target="_blank">📎 Source: {url}</a>' for url in source_urls)
                answer += f"<br><br><strong>Sources:</strong><br>{joined_sources}"
            else:
                joined_sources = "<br>".join(f'<a href="{url}" target="_blank">📎 Lähde: {url}</a>' for url in source_urls)
                answer += f"<br><br><strong>Lähteet:</strong><br>{joined_sources}"
        
        # Update server-side chat_history for persistent display and future context
        chat_history.append({'type': 'user', 'content': user_question})
        chat_history.append({'type': 'assistant', 'content': answer})

        return jsonify({
            'answer': answer,
            'sources': list(source_urls),
            'language': original_lang
        })
        
    except Exception as e:
        print(f"Error processing question: {e}")
        traceback.print_exc() # Print full traceback for debugging
        # Append an error message to chat_history for visibility on refresh
        error_msg = f'Sorry, an internal error occurred: {str(e)}'
        chat_history.append({'type': 'user', 'content': user_question}) # Still log user's attempt
        chat_history.append({'type': 'assistant', 'content': error_msg})
        return jsonify({
            'error': error_msg,
            'answer': error_msg # Send error back to client to display
        }), 500


@app.route('/search_duodecim', methods=['POST'])
def search_duodecim_route():
    """
    Triggers a search using the Duodecim scraper based on the last user question.
    This route currently causes a redirect, which will refresh the page.
    For a fully AJAX experience, this should also return JSON and be handled client-side.
    """
    global chat_history

    # Find the last user question from the server's chat_history
    last_question = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), None)
    
    if not last_question:
        # If no question, provide feedback to the user.
        # For an AJAX call, you'd return jsonify({'error': ...}), 400
        # For a redirect, we add it to history and then redirect.
        chat_history.append({'type': 'assistant', 'content': 'Please ask a question first before searching Duodecim.'})
        return redirect(url_for('chat'))

    try:
        # Attempt to import the scraper. Assumes duodecim_scraper.py exists.
        from duodecim_scraper import search_duodecim
        
        print(f"Searching Duodecim for: {last_question}")
        result = search_duodecim(last_question, headless=True) # Run in headless mode
        
        duodecim_answer = f"<strong>🔍 Duodecim AI:</strong><br>{result.get('answer', 'No answer found from Duodecim.')}"

        unique_links = {}
        for src in result.get('sources', []):
            key = (src.get('title', 'No Title'), src.get('url', '#'))
            if key not in unique_links:
                unique_links[key] = f'<a href="{key[1]}" target="_blank">📎 {key[0]}: {key[1]}</a>'

        if unique_links:
            duodecim_answer += "<br><br><strong>SOURCES:</strong><br>" + "<br>".join(unique_links.values())

        chat_history.append({'type': 'assistant', 'content': duodecim_answer})

    except ImportError:
        error_msg = "<strong>Duodecim AI:</strong> The 'duodecim_scraper.py' module was not found. Please ensure it's in the same directory."
        chat_history.append({'type': 'assistant', 'content': error_msg})
        print(error_msg)
    except Exception as e:
        error_msg = f"<strong>Duodecim AI:</strong> An error occurred while fetching the answer: {e}"
        chat_history.append({'type': 'assistant', 'content': error_msg})
        print(f"Error during Duodecim search: {e}")
        traceback.print_exc()

    return redirect(url_for('chat')) # Redirect to refresh chat with new message

@app.route('/clear_chat')
def clear_chat():
    """Clears the server-side chat history and redirects to the chat page."""
    global chat_history, conversation_chain
    chat_history = []
    # Also clear the memory of the conversation chain
    if conversation_chain and hasattr(conversation_chain, 'memory'):
        conversation_chain.memory.clear()
        print("Conversation chain memory cleared.")
    print("Chat history cleared.")
    return redirect(url_for('chat'))

@app.route('/load/uniapnea')
def load_uniapnea_folder():
    """
    Loads data into the vectorstore and initializes the conversation chain.
    This route is meant to be called once to set up the system.
    """
    global vectorstore, conversation_chain, chat_history
    
    # Prevent re-loading if already loaded
    if vectorstore is not None:
        print("⚠️ Vectorstore already loaded, skipping reload and clearing chat.")
        chat_history = [] # Clear chat if re-visiting this route after init
        if conversation_chain and hasattr(conversation_chain, 'memory'):
            conversation_chain.memory.clear()
        return redirect(url_for('chat'))
        
    # IMPORTANT: Update this path to where your PDF and JSON files are located!
    # For example: folder_path = os.path.join(os.getcwd(), "sources")
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    
    try:
        print(f"Attempting to load vectorstore from: {folder_path}...")
        vectorstore = build_vectorstore_from_sources(folder_path)
        conversation_chain = get_conversation_chain(vectorstore)
        chat_history = [{'type': 'assistant', 'content': 'Hello! I am your Ensitietoa Chatbot. I can answer questions about COPD, sleep apnea, and related respiratory health topics. How can I help you today?'}]
        print("✅ Successfully loaded vectorstore and initialized chatbot!")
        return redirect(url_for('chat'))
        
    except Exception as e:
        print(f"❌ Error loading vectorstore: {e}")
        traceback.print_exc() # Print full Python traceback for debugging
        return f"<h1>Error Loading Data</h1><p>There was an error loading the data: {str(e)}</p><p>Please ensure the '{folder_path}' folder exists and contains valid PDF/JSON files, and your OpenAI API key is set.</p><a href='/'>Go back to home</a>", 500

# --- Health Check and Error Handlers ---

@app.route('/health')
def health():
    """Endpoint for health checking the application."""
    return jsonify({
        'status': 'ok',
        'vectorstore_loaded': vectorstore is not None,
        'conversation_chain_ready': conversation_chain is not None
    })

@app.errorhandler(400)
def bad_request(e):
    """Custom error handler for 400 Bad Request."""
    print(f"Bad request error: {e}")
    return jsonify({'error': 'Bad request', 'message': str(e)}), 400

@app.errorhandler(413)
def request_entity_too_large(e):
    """Custom error handler for 413 Request Entity Too Large."""
    return jsonify({'error': 'Request too large', 'message': str(e)}), 413

@app.errorhandler(500)
def internal_error(e):
    """Custom error handler for 500 Internal Server Error."""
    print(f"Internal server error: {e}")
    traceback.print_exc() # Print traceback for internal errors
    return jsonify({'error': 'Internal server error', 'message': str(e)}), 500

# --- Main Execution Block ---

if __name__ == '__main__':
    # Check for OpenAI API key environment variable on startup
    if not os.getenv('OPENAI_API_KEY'):
        print("="*80)
        print("⚠️ WARNING: OPENAI_API_KEY environment variable not set!")
        print("Please set it using: export OPENAI_API_KEY='your-api-key-here' (Linux/macOS) or")
        print("                     set OPENAI_API_KEY=your-api-key-here (Windows Cmd) or")
        print("                     $env:OPENAI_API_KEY='your-api-key-here' (Windows PowerShell)")
        print("The application will not function correctly without the API key.")
        print("="*80)
    
    # Define the folder path for your source documents
    # IMPORTANT: Adjust this path to your actual directory!
    source_folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    
    # Attempt to load data on application startup
    if os.path.exists(source_folder_path):
        try:
            print(f"Attempting to load vectorstore on startup from: {source_folder_path}...")
            vectorstore = build_vectorstore_from_sources(source_folder_path)
            conversation_chain = get_conversation_chain(vectorstore)
            chat_history.append({'type': 'assistant', 'content': 'Hello! I am your Ensitietoa Chatbot. I can answer questions about COPD, sleep apnea, and related respiratory health topics. How can I help you today?'})
            print("📁 Folder and .json loaded successfully on startup!")
        except Exception as e:
            print(f"⚠️ Error loading folder on startup: {e}")
            traceback.print_exc()
            print("The app will start without pre-loaded data. Please manually trigger loading via /load/uniapnea or fix the path/data.")
    else:
        print(f"⚠️ Source folder path does not exist: {source_folder_path}")
        print("The app will start without pre-loaded data. Please ensure the path is correct or manually trigger loading via /load/uniapnea.")
            
    app.run(debug=False, port=5000) 


Attempting to load vectorstore on startup from: C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



...

Translated answer back to English: Based on my knowledge, I cannot answer this question....

=== Processing question: what is COPD? ===
Detected language: en
Translated to Finnish for processing: The translation of "what is COPD?" to Finnish is "mikä on COPD?"
Querying conversation chain...
Received 15 sources.
Raw answer preview: COPD (Chronic Obstructive Pulmonary Disease) on kansainvälinen kattotermi, joka viittaa krooniseen keuhkoahtaumatautiin. Tämä sairaus sisältää erilaisia keuhkosairauksia, joista yksi tunnetuimmista on...


127.0.0.1 - - [30/Jun/2025 13:53:21] "POST /process_question HTTP/1.1" 200 -


Translated answer back to English: COPD (Chronic Obstructive Pulmonary Disease) is an international umbrella term that refers to chronic obstructive pulmonary disease. This condition includes various lung diseases, one of the most well...

=== Processing question: What is sleep apnea? ===
Detected language: en
Translated to Finnish for processing: Mikä on uniapnea?
Querying conversation chain...
Received 15 sources.
Raw answer preview: Uniapnea on unenaikainen hengityshäiriö, jossa hengitys katkeaa toistuvasti unen aikana. Tämä häiritsee unen laatua ja kuormittaa elimistöä. Tyypillisiä oireita ovat kuorsaus, hengityskatkokset, heräi...


127.0.0.1 - - [30/Jun/2025 13:53:53] "POST /process_question HTTP/1.1" 200 -


Translated answer back to English: Sleep apnea is a temporary respiratory disorder in which breathing repeatedly stops during sleep. This disrupts the quality of sleep and puts strain on the body. Typical symptoms include snoring, brea...


In [ ]:
import os
import json
import glob
from flask import Flask, render_template, request, redirect, url_for, jsonify
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain import PromptTemplate
from langdetect import detect
import traceback

# Flask App
app = Flask(__name__)

app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024 # 16MB max request size
app.config['WTF_CSRF_ENABLED'] = False

vectorstore = None
conversation_chain = None
chat_history = []

def load_pdfs_from_folder(folder_path):
    pdf_paths = glob.glob(os.path.join(folder_path, '*.pdf'))
    pdf_files = [open(path, 'rb') for path in pdf_paths]
    print(f"Loaded {len(pdf_files)} PDF files from {folder_path}")
    return pdf_files

def load_json_chunks(file_path):
    if not os.path.exists(file_path):
        print(f"JSON file not found: {file_path}")
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        print(f"Loaded {len(data)} JSON chunks from {file_path}")
        return data

def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        try:
            pdf_reader = PdfReader(pdf)
            for page in pdf_reader.pages:
                content = page.extract_text()
                if content:
                    text += content
        except Exception as e:
            print(f"Error reading PDF {pdf.name}: {e}")
            continue
    return text

def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500,
        chunk_overlap=100,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    return chunks

def create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=100):
    vectorstores = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    print(f"Starting to create embeddings in {total_batches} batches...")
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_metadatas = metadatas[i:i + batch_size]
        batch_num = i // batch_size + 1

        print(f"Processing batch {batch_num}/{total_batches} ({batch_num/total_batches*100:.1f}%) with {len(batch_texts)} chunks...")

        try:
            batch_vectorstore = FAISS.from_texts(
                texts=batch_texts,
                embedding=embeddings,
                metadatas=batch_metadatas
            )
            vectorstores.append(batch_vectorstore)
        except Exception as e:
            print(f"Error processing batch {batch_num}: {e}")
            continue

    print(f"✅ Successfully processed {len(vectorstores)}/{total_batches} batches.")

    if vectorstores:
        print("🔗 Merging vectorstores...")
        main_vectorstore = vectorstores[0]
        for vs in vectorstores[1:]:
            main_vectorstore.merge_from(vs)
        print("✅ Merged all vectorstores.")
        return main_vectorstore
    else:
        raise Exception("No vectorstores were created successfully. Check input data and API key.")

def build_vectorstore_from_sources(folder_path):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    all_chunks_data = []

    # Process PDFs
    pdf_files = load_pdfs_from_folder(folder_path)
    if pdf_files:
        print("Extracting text from PDF files...")
        pdf_text = get_pdf_text(pdf_files)
        print(f"Total extracted PDF text length: {len(pdf_text)} characters")

        pdf_chunks = get_text_chunks(pdf_text)
        print(f"Created {len(pdf_chunks)} PDF chunks.")

        for chunk in pdf_chunks:
            all_chunks_data.append({"text": chunk, "metadata": {"source": "PDF"}})

    # Process JSON Chunks
    json_path = os.path.join(folder_path, "uniapnea_chunks.json")
    if os.path.exists(json_path):
        print(f"Loading JSON chunks from {json_path}...")
        json_chunks = load_json_chunks(json_path)

        for item in json_chunks:
            source_url = item.get("source", "Unknown")
            all_chunks_data.append({"text": item["text"], "metadata": {"source": source_url}})
    else:
        print(f"JSON chunk file not found at {json_path}. Skipping JSON loading.")

    print(f"Total chunks collected from all sources: {len(all_chunks_data)}")

    if not all_chunks_data:
        raise Exception("No content found to process for vectorstore creation. Check PDF folder and JSON file path.")

    texts = [x["text"] for x in all_chunks_data]
    metadatas = [x["metadata"] for x in all_chunks_data]

    return create_embeddings_in_batches(texts, metadatas, embeddings)

def get_conversation_chain(vectorstore):
    llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7)

    friendly_prompt = PromptTemplate(
        input_variables=["question", "context", "chat_history"],
        template="""
You are a helpful assistant. You only answer based on the provided source materials.

RULES:
- If the user asks for a comparison (e.g., "difference between", "compare", "vs"), answer by comparing them.
- If only one topic is asked, only talk about that - do not mention other topics.
- If the context contains information about the requested topic(s), answer based on it.
- If the question is in English, answer in English.
- If the question is in Finnish, answer in Finnish.
- If the context does not contain information about the requested topic, say: "Based on my knowledge, I cannot answer this question." (or the Finnish equivalent).

Here is the previous conversation history:
{chat_history}

Here is the user's question:
{question}

Here is the context relevant to the question:
{context}

Answer:
"""
    )

    memory = ConversationBufferMemory(
        memory_key='chat_history',
        return_messages=True,
        output_key='answer'
    )

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": 15}
        ),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": friendly_prompt},
        output_key="answer"
    )

def translate_to_finnish(question):
    try:
        llm = ChatOpenAI(model_name="gpt-4o-mini")
        prompt = f"Translate the following question to Finnish:\n\n{question}"
        return llm.predict(prompt).strip()
    except Exception as e:
        print(f"Translation error (to Finnish): {e}")
        return question

def translate_to_english(answer):
    try:
        llm = ChatOpenAI(model_name="gpt-4o-mini")
        prompt = f"Translate the following answer to English:\n\n{answer}"
        return llm.predict(prompt).strip()
    except Exception as e:
        print(f"Translation error (to English): {e}")
        return answer

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/chat', methods=['GET'])
def chat():
    return render_template('chat.html', chat_history=chat_history)

@app.route('/process_question', methods=['POST'])
def process_question():
    global vectorstore, conversation_chain, chat_history

    try:
        data = request.get_json()
        if not data or 'question' not in data:
            return jsonify({'error': 'No question provided'}), 400

        user_question = data['question'].strip()
        if not user_question:
            return jsonify({'error': 'Empty question'}), 400

        print(f"\n=== Processing question: {user_question} ===")

        if conversation_chain is None:
            return jsonify({'error': 'System not initialized. Please load data first.'}), 500

        # Language detection for input question
        try:
            lang = detect(user_question)
            if lang not in ['en', 'fi'] or len(user_question.strip()) < 10:
                english_words = ['what', 'is', 'how', 'can', 'you', 'explain', 'more', 'about', 'it', 'the', 'this', 'that']
                finnish_words = ['mikä', 'mitä', 'miten', 'voitko', 'selitä', 'lisää', 'tästä', 'kuinka', 'on', 'kerro']
                user_lower = user_question.lower()
                english_count = sum(1 for word in english_words if word in user_lower)
                finnish_count = sum(1 for word in finnish_words if word in user_lower)
                lang = 'fi' if finnish_count > english_count else 'en'
        except:
            lang = 'en'

        print(f"Detected language: {lang}")

        original_lang = lang
        if lang == 'en':
            used_question = translate_to_finnish(user_question)
            print(f"Translated to Finnish for processing: {used_question}")
        else:
            used_question = user_question

        print("Querying conversation chain...")
        response = conversation_chain({'question': used_question})

        answer = response.get('answer', 'Tietojeni perusteella en osaa vastata tähän kysymykseen.')
        sources = response.get('source_documents', [])
        source_urls = set()

        print(f"Received {len(sources)} sources.")
        print(f"Raw answer preview: {answer[:200]}...")

        for doc in sources:
            if "source" in doc.metadata and doc.metadata["source"] != "PDF":
                clean_url = doc.metadata["source"].split("#")[0]
                source_urls.add(clean_url)

        # Translate the answer back to English if the original question was in English
        if original_lang == 'en' and answer:
            answer = translate_to_english(answer)
            print(f"Translated answer back to English: {answer[:200]}...")

        # ======= THIS IS THE FIX =======
        # Do NOT append sources to the answer!
        # Just return them as a separate field for the frontend.

        chat_history.append({'type': 'user', 'content': user_question})
        chat_history.append({'type': 'assistant', 'content': answer, 'sources': list(source_urls), 'language': original_lang})

        return jsonify({
            'answer': answer,              # pure answer text, NO links here!
            'sources': list(source_urls),  # sources only in this array
            'language': original_lang
        })

    except Exception as e:
        print(f"Error processing question: {e}")
        traceback.print_exc()
        error_msg = f'Sorry, an internal error occurred: {str(e)}'
        chat_history.append({'type': 'user', 'content': user_question})
        chat_history.append({'type': 'assistant', 'content': error_msg})
        return jsonify({
            'error': error_msg,
            'answer': error_msg
        }), 500

@app.route('/search_duodecim', methods=['POST'])
def search_duodecim_route():
    global chat_history

    last_question = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), None)
    if not last_question:
        chat_history.append({'type': 'assistant', 'content': 'Please ask a question first before searching Duodecim.'})
        return redirect(url_for('chat'))

    try:
        from duodecim_scraper import search_duodecim
        print(f"Searching Duodecim for: {last_question}")
        result = search_duodecim(last_question, headless=True)
        duodecim_answer = f"<strong>🔍 Duodecim AI:</strong><br>{result.get('answer', 'No answer found from Duodecim.')}"
        unique_links = {}
        for src in result.get('sources', []):
            key = (src.get('title', 'No Title'), src.get('url', '#'))
            if key not in unique_links:
                unique_links[key] = f'<a href="{key[1]}" target="_blank">📎 {key[0]}: {key[1]}</a>'
        if unique_links:
            duodecim_answer += "<br><br><strong>SOURCES:</strong><br>" + "<br>".join(unique_links.values())
        chat_history.append({'type': 'assistant', 'content': duodecim_answer})
    except ImportError:
        error_msg = "<strong>Duodecim AI:</strong> The 'duodecim_scraper.py' module was not found. Please ensure it's in the same directory."
        chat_history.append({'type': 'assistant', 'content': error_msg})
        print(error_msg)
    except Exception as e:
        error_msg = f"<strong>Duodecim AI:</strong> An error occurred while fetching the answer: {e}"
        chat_history.append({'type': 'assistant', 'content': error_msg})
        print(f"Error during Duodecim search: {e}")
        traceback.print_exc()

    return redirect(url_for('chat'))

@app.route('/clear_chat')
def clear_chat():
    global chat_history, conversation_chain
    chat_history = []
    if conversation_chain and hasattr(conversation_chain, 'memory'):
        conversation_chain.memory.clear()
        print("Conversation chain memory cleared.")
    print("Chat history cleared.")
    return redirect(url_for('chat'))

@app.route('/load/uniapnea')
def load_uniapnea_folder():
    global vectorstore, conversation_chain, chat_history

    if vectorstore is not None:
        print("⚠️ Vectorstore already loaded, skipping reload and clearing chat.")
        chat_history = []
        if conversation_chain and hasattr(conversation_chain, 'memory'):
            conversation_chain.memory.clear()
        return redirect(url_for('chat'))

    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"

    try:
        print(f"Attempting to load vectorstore from: {folder_path}...")
        vectorstore = build_vectorstore_from_sources(folder_path)
        conversation_chain = get_conversation_chain(vectorstore)
        chat_history = [{'type': 'assistant', 'content': 'Hello! I am your Ensitietoa Chatbot. I can answer questions about COPD, sleep apnea, and related respiratory health topics. How can I help you today?'}]
        print("✅ Successfully loaded vectorstore and initialized chatbot!")
        return redirect(url_for('chat'))
    except Exception as e:
        print(f"❌ Error loading vectorstore: {e}")
        traceback.print_exc()
        return f"<h1>Error Loading Data</h1><p>There was an error loading the data: {str(e)}</p><p>Please ensure the '{folder_path}' folder exists and contains valid PDF/JSON files, and your OpenAI API key is set.</p><a href='/'>Go back to home</a>", 500

@app.route('/health')
def health():
    return jsonify({
        'status': 'ok',
        'vectorstore_loaded': vectorstore is not None,
        'conversation_chain_ready': conversation_chain is not None
    })

@app.errorhandler(400)
def bad_request(e):
    print(f"Bad request error: {e}")
    return jsonify({'error': 'Bad request', 'message': str(e)}), 400

@app.errorhandler(413)
def request_entity_too_large(e):
    return jsonify({'error': 'Request too large', 'message': str(e)}), 413

@app.errorhandler(500)
def internal_error(e):
    print(f"Internal server error: {e}")
    traceback.print_exc()
    return jsonify({'error': 'Internal server error', 'message': str(e)}), 500

if __name__ == '__main__':
    if not os.getenv('OPENAI_API_KEY'):
        print("="*80)
        print("⚠️ WARNING: OPENAI_API_KEY environment variable not set!")
        print("Please set it using: export OPENAI_API_KEY='your-api-key-here' (Linux/macOS) or")
        print("                     set OPENAI_API_KEY=your-api-key-here (Windows Cmd) or")
        print("                     $env:OPENAI_API_KEY='your-api-key-here' (Windows PowerShell)")
        print("The application will not function correctly without the API key.")
        print("="*80)

    source_folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"

    if os.path.exists(source_folder_path):
        try:
            print(f"Attempting to load vectorstore on startup from: {source_folder_path}...")
            vectorstore = build_vectorstore_from_sources(source_folder_path)
            conversation_chain = get_conversation_chain(vectorstore)
            chat_history.append({'type': 'assistant', 'content': 'Hello! I am your Ensitietoa Chatbot. I can answer questions about COPD, sleep apnea, and related respiratory health topics. How can I help you today?'})
            print("📁 Folder and .json loaded successfully on startup!")
        except Exception as e:
            print(f"⚠️ Error loading folder on startup: {e}")
            traceback.print_exc()
            print("The app will start without pre-loaded data. Please manually trigger loading via /load/uniapnea or fix the path/data.")
    else:
        print(f"⚠️ Source folder path does not exist: {source_folder_path}")
        print("The app will start without pre-loaded data. Please ensure the path is correct or manually trigger loading via /load/uniapnea.")

    app.run(debug=False, port=5000)


Attempting to load vectorstore on startup from: C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources...



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



...

Received 15 sources.
Raw answer preview: Uniapnea on unenaikainen hengityshäiriö, jossa hengitys katkeaa toistuvasti unen aikana. Tämä häiritsee unen laatua ja kuormittaa elimistöä. Uniapnea jaotellaan neljään eri muotoon, joista yleisin on ...


127.0.0.1 - - [01/Jul/2025 09:32:38] "POST /process_question HTTP/1.1" 200 -


Translated answer back to English: Sleep apnea is a temporary breathing disorder in which breathing repeatedly stops during sleep. This disrupts the quality of sleep and puts a strain on the body. Sleep apnea is classified into four di...

=== Processing question: what is COPD? ===
Detected language: en
Translated to Finnish for processing: The translation of "what is COPD?" to Finnish is "Mikä on COPD?"
Querying conversation chain...
Received 15 sources.
Raw answer preview: COPD (krooninen obstruktiivinen keuhkosairaus) on kansainvälinen kattotermi, joka viittaa keuhkoahtaumatautiin. Se sisältää keuhkoputkentulehduksen ja emfyseeman, jotka molemmat vaikuttavat hengityste...


127.0.0.1 - - [01/Jul/2025 09:33:43] "POST /process_question HTTP/1.1" 200 -


Translated answer back to English: COPD (chronic obstructive pulmonary disease) is an international umbrella term that refers to chronic bronchitis and emphysema, both of which affect the function of the airways. In emphysema, the alve...


## Version 2 (Limited Questions)

In [33]:
# Flask App V2 - Limited Questions
app = Flask(__name__)

# Global state
vectorstore = None
conversation_chain = None
chat_history = []
last_user_question = ""
question_count = 0  # NEW: Track question count
MAX_QUESTIONS = 3   # NEW: Maximum allowed questions
COURSE_LINK = "https://your-course-website.com"  # NEW: Link to your courses

# Load all PDFs from a folder
def load_pdfs_from_folder(folder_path):
    pdf_paths = glob.glob(os.path.join(folder_path, '*.pdf'))
    pdf_files = [open(path, 'rb') for path in pdf_paths]
    return pdf_files

# Load JSON chunks with source metadata
def load_json_chunks(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Translate English question to Finnish
def translate_to_finnish(question):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")
    prompt = f"Translate the following question to Finnish:\n\n{question}"
    return llm.predict(prompt).strip()

# Translate Finnish answer to English
def translate_to_english(answer):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")
    prompt = f"Translate the following answer to English:\n\n{answer}"
    return llm.predict(prompt).strip()

# NEW: Check if user has reached question limit
def check_question_limit():
    global question_count
    return question_count >= MAX_QUESTIONS

# NEW: Get limit message
def get_limit_message(lang='en'):
    if lang == 'fi':
        return f"""
        <div class="bg-yellow-100 border-l-4 border-yellow-500 p-4 rounded-lg">
            <h3 class="font-bold text-yellow-800">Kokeiluversio päättynyt</h3>
            <p class="text-yellow-700">Olet käyttänyt kaikki {MAX_QUESTIONS} ilmaista kysymystä.</p>
            <p class="text-yellow-700 mt-2">Saadaksesi pääsyn täydelliseen versioon ja unlimited kysymyksiin, liity kurssillemme:</p>
            <a href="{COURSE_LINK}" target="_blank" class="inline-block mt-3 px-4 py-2 bg-blue-600 text-white rounded-lg hover:bg-blue-700">
                📚 Liity kurssille täällä
            </a>
        </div>
        """
    else:
        return f"""
        <div class="bg-yellow-100 border-l-4 border-yellow-500 p-4 rounded-lg">
            <h3 class="font-bold text-yellow-800">Trial Version Ended</h3>
            <p class="text-yellow-700">You have used all {MAX_QUESTIONS} free questions.</p>
            <p class="text-yellow-700 mt-2">To get access to the full version with unlimited questions, join our course:</p>
            <a href="{COURSE_LINK}" target="_blank" class="inline-block mt-3 px-4 py-2 bg-blue-600 text-white rounded-lg hover:bg-blue-700">
                📚 Join our course here
            </a>
        </div>
        """

# Chat route - UPDATED WITH QUESTION LIMIT
@app.route('/chat', methods=['GET', 'POST'])
def chat():
    global vectorstore, conversation_chain, chat_history, last_user_question, question_count

    if request.method == 'POST':
        user_question = request.form['user_question']
        last_user_question = user_question
        
        # NEW: Check question limit before processing
        if check_question_limit():
            # Better language detection for limit message
            try:
                lang = detect(user_question)
                if lang not in ['en', 'fi']:
                    lang = 'en'  # Default to English
            except:
                lang = 'en'
            
            # Add the limit message to chat history
            limit_msg = get_limit_message(lang)
            chat_history.append({'type': 'user', 'content': user_question})
            chat_history.append({'type': 'assistant', 'content': limit_msg})
            return render_template('chat2.html', chat_history=chat_history, questions_remaining=0)
        
        # Increment question count
        question_count += 1
        
        # Better language detection with fallbacks - FIXED
        try:
            lang = detect(user_question)
            # Fix common misdetections
            if lang not in ['en', 'fi'] or len(user_question.strip()) < 10:
                # For short phrases, check if it's obviously English or Finnish
                english_words = ['what', 'is', 'how', 'can', 'you', 'explain', 'more', 'about', 'it', 'the', 'this', 'that']
                finnish_words = ['mikä', 'mitä', 'miten', 'voitko', 'selitä', 'lisää', 'tästä', 'kuinka', 'on', 'kerro']
                
                user_lower = user_question.lower()
                english_count = sum(1 for word in english_words if word in user_lower)
                finnish_count = sum(1 for word in finnish_words if word in user_lower)
                
                if finnish_count > english_count:
                    lang = 'fi'
                elif english_count > 0:
                    lang = 'en'
                else:
                    lang = 'en'  # Default to English for unclear cases
        except:
            lang = 'en'  # Default fallback
        
        print(f"Language detection: '{user_question}' -> {lang}")
        print(f"Question {question_count}/{MAX_QUESTIONS}")  # NEW: Debug info

        # Simple context enhancement like your original - only for very short questions
        original_question = user_question  # Store original before any modification
        if len(user_question.strip().split()) < 3:  # Back to < 3 like your original
            last_user_msg = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), "")
            user_question = f"{last_user_msg}\n{user_question}"
            print(f"Enhanced question: {user_question[:100]}...")

        # Translate to Finnish for processing, but remember original language
        original_lang = lang
        if lang == 'en':
            used_question = translate_to_finnish(user_question)
        else:
            used_question = user_question

        response = conversation_chain({'question': used_question})
        sources = response.get('source_documents', [])
        source_urls = set()

        # DEBUG: Print what sources were found
        print(f"\n=== DEBUG INFO ===")
        print(f"Original question: {original_question}")
        print(f"Detected language: {original_lang}")
        print(f"Used question: {used_question}")
        print(f"Found {len(sources)} source documents")
        for i, doc in enumerate(sources[:3]):  # Show first 3 sources
            print(f"Source {i+1}: {doc.page_content[:200].replace(chr(10), ' ')}")
            print(f"Metadata: {doc.metadata}")
            print("---")
        print(f"Raw answer: {response.get('answer', 'No answer')[:300]}...")
        print(f"==================\n")

        # Check if we have relevant sources and good content
        if not sources or all(not doc.page_content.strip() for doc in sources):
            answer = "Tietojeni perusteella en osaa vastata tähän kysymykseen."
        else:
            answer = response.get('answer', "Tietojeni perusteella en osaa vastata tähän kysymykseen.")
            
            # Additional check: if answer seems to be refusing despite having sources
            if ("en osaa vastata" in answer or "cannot answer" in answer) and len(sources) > 0:
                # Check if any source actually contains relevant content
                relevant_content = []
                search_terms = used_question.lower().split()
                for doc in sources:
                    content_lower = doc.page_content.lower()
                    if any(term in content_lower for term in search_terms if len(term) > 3):
                        relevant_content.append(doc.page_content[:200])
                
                if relevant_content:
                    print(f"Found relevant content but AI refused: {relevant_content[0][:100]}...")

        for doc in sources:
            if "source" in doc.metadata and doc.metadata["source"] != "PDF":
                clean_url = doc.metadata["source"].split("#")[0]
                source_urls.add(clean_url)

        if lang == 'en':
            answer = translate_to_english(answer)
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Source: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"
        else:
            if source_urls:
                joined = "<br>".join(f'<a href="{url}" target="_blank">📎 Lähde: {url}</a>' for url in source_urls)
                answer += f"<br><br>{joined}"

        chat_history.append({'type': 'user', 'content': original_question})
        chat_history.append({'type': 'assistant', 'content': answer})

    # NEW: Calculate remaining questions
    questions_remaining = max(0, MAX_QUESTIONS - question_count)
    return render_template('chat2.html', chat_history=chat_history, questions_remaining=questions_remaining)

@app.route('/search_duodecim', methods=['POST'])
def search_duodecim_route():
    global chat_history

    last_question = next((m['content'] for m in reversed(chat_history) if m['type'] == 'user'), None)
    if not last_question:
        return redirect(url_for('chat'))

    try:
        from duodecim_scraper import search_duodecim
        result = search_duodecim(last_question, headless=True)
        duodecim_answer = f"<strong>🔍 Duodecim AI:</strong><br>{result['answer']}"

        unique_links = {}
        for src in result['sources']:
            key = (src['title'], src['url'])
            if key not in unique_links:
                unique_links[key] = f'<a href="{src["url"]}" target="_blank">📎 {src["title"]}: {src["url"]}</a>'

        if unique_links:
            duodecim_answer += "<br><br><strong>SOURCES:</strong><br>" + "<br>".join(unique_links.values())

        chat_history.append({'type': 'assistant', 'content': duodecim_answer})

    except Exception as e:
        chat_history.append({'type': 'assistant', 'content': f"<strong>Duodecim AI:</strong> An error occurred while fetching the answer. {e}"})

    return redirect(url_for('chat'))

# Extract raw text
def get_pdf_text(pdf_docs):
    text = ""
    for pdf in pdf_docs:
        pdf_reader = PdfReader(pdf)
        for page in pdf_reader.pages:
            content = page.extract_text()
            if content:
                text += content
    return text

# Split text into smaller chunks - REDUCED CHUNK SIZE
def get_text_chunks(text):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=500,  # Reduced from 1000
        chunk_overlap=100,  # Reduced from 200
        length_function=len
    )
    return text_splitter.split_text(text)

# NEW: Process embeddings in batches to avoid token limits
def create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=50):
    """Process embeddings in smaller batches to avoid token limits"""
    vectorstores = []
    total_batches = (len(texts) + batch_size - 1) // batch_size
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        batch_metadatas = metadatas[i:i + batch_size]
        batch_num = i // batch_size + 1
        
        # Progress indicator every 25 batches to reduce spam
        if batch_num % 25 == 0 or batch_num == 1 or batch_num == total_batches:
            print(f"Processing batch {batch_num}/{total_batches} ({batch_num/total_batches*100:.1f}%)")
        
        try:
            # Create vectorstore for this batch
            batch_vectorstore = FAISS.from_texts(
                texts=batch_texts, 
                embedding=embeddings, 
                metadatas=batch_metadatas
            )
            vectorstores.append(batch_vectorstore)
            
        except Exception as e:
            print(f"Error processing batch {batch_num}: {e}")
            # Skip this batch and continue
            continue
    
    print(f"✅ Successfully processed {len(vectorstores)}/{total_batches} batches")
    
    # Merge all vectorstores
    if vectorstores:
        print("🔗 Merging vectorstores...")
        main_vectorstore = vectorstores[0]
        for vs in vectorstores[1:]:
            main_vectorstore.merge_from(vs)
        return main_vectorstore
    else:
        raise Exception("No vectorstores were created successfully")

# Build the vectorstore from both PDFs and JSON chunks - UPDATED WITH BATCH PROCESSING
def build_vectorstore_from_sources(folder_path):
    embeddings = OpenAIEmbeddings()
    all_chunks = []

    # Add PDF chunks
    pdf_files = load_pdfs_from_folder(folder_path)
    if pdf_files:
        print("Processing PDF files...")
        pdf_text = get_pdf_text(pdf_files)
        print(f"Total PDF text length: {len(pdf_text)} characters")
        
        pdf_chunks = get_text_chunks(pdf_text)
        print(f"Created {len(pdf_chunks)} PDF chunks")
        
        for chunk in pdf_chunks:
            all_chunks.append({"text": chunk, "metadata": {"source": "PDF"}})

    # Add JSON chunks (from scraper)
    json_path = os.path.join(folder_path, "uniapnea_chunks.json")
    if os.path.exists(json_path):
        print("Processing JSON chunks...")
        json_chunks = load_json_chunks(json_path)
        print(f"Loaded {len(json_chunks)} JSON chunks")
        
        for item in json_chunks:
            all_chunks.append({"text": item["text"], "metadata": {"source": item["source"]}})

    print(f"Total chunks to process: {len(all_chunks)}")
    
    if not all_chunks:
        raise Exception("No content found to process")

    texts = [x["text"] for x in all_chunks]
    metadatas = [x["metadata"] for x in all_chunks]

    # Use batch processing instead of processing all at once (larger batches = faster)
    return create_embeddings_in_batches(texts, metadatas, embeddings, batch_size=100)

# Conversation chain builder with friendly tone
def get_conversation_chain(vectorstore):
    llm = ChatOpenAI(model_name="gpt-4.1-nano")

    friendly_prompt = PromptTemplate(
        input_variables=["question", "context", "chat_history"],
        template="""
Olet avulias assistentti. Vastaat vain annettujen lähdemateriaalien perusteella.

SÄÄNNÖT:
- Jos käyttäjä kysyy vertailua (esim. "difference between", "compare", "vs"), vastaa vertaillen
- Jos kysytään vain yhdestä aiheesta, kerro vain siitä - älä mainitse muita aiheita
- Jos asiayhteys sisältää tietoa kysytystä aiheesta/aiheista, vastaa sen perusteella  
- Jos kysymys on englanniksi, vastaa englanniksi
- Jos kysymys on suomeksi, vastaa suomeksi
- Jos asiayhteys ei sisällä tietoa kysytystä aiheesta, sano: "Tietojeni perusteella en osaa vastata tähän kysymykseen."

Tässä keskustelun aiempi sisältö:
{chat_history}

Tässä käyttäjän kysymys:
{question}

Tässä asiayhteys:
{context}

Vastaus:
"""
    )

    memory = ConversationBufferMemory(
        memory_key='chat_history',
        return_messages=True,
        output_key='answer'
    )

    return ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectorstore.as_retriever(
            search_type="similarity", 
            search_kwargs={"k": 15}  # Get more chunks, no threshold restriction
        ),
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": friendly_prompt},
        output_key="answer"
    )

# NEW: Reset question count (for testing or admin use)
@app.route('/reset_questions')
def reset_questions():
    global question_count
    question_count = 0
    return redirect(url_for('chat'))

# Clear chat button route - UPDATED TO RESET QUESTIONS
@app.route('/clear_chat')
def clear_chat():
    global chat_history, question_count
    chat_history = []
    question_count = 0  # NEW: Reset question count when clearing chat
    return redirect(url_for('chat'))

# Home page
@app.route('/')
def index():
    return render_template('index.html')

@app.route('/load/uniapnea')
def load_uniapnea_folder():
    global vectorstore, conversation_chain, chat_history
    
    # Check if already loaded
    if vectorstore is not None:
        print("⚠️ Vectorstore already loaded, skipping reload")
        chat_history = []  # Just clear chat history
        return redirect(url_for('chat'))
    
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    
    try:
        print("Starting to load vectorstore...")
        vectorstore = build_vectorstore_from_sources(folder_path)
        conversation_chain = get_conversation_chain(vectorstore)
        chat_history = []
        print("✅ Successfully loaded vectorstore!")
        return redirect(url_for('chat'))
        
    except Exception as e:
        print(f"❌ Error loading vectorstore: {e}")
        return f"<h1>Error Loading Data</h1><p>Error: {str(e)}</p><a href='/'>Go back to home</a>", 500

# Load folder on startup - UPDATED WITH BETTER ERROR HANDLING
if __name__ == '__main__':
    folder_path = r"C:\Users\maryamata\OneDrive - KamIT 365\KAMK\Ensitietoa\sources"
    if os.path.exists(folder_path):
        try:
            print("Loading vectorstore on startup...")
            vectorstore = build_vectorstore_from_sources(folder_path)
            conversation_chain = get_conversation_chain(vectorstore)
            print("📁 Folder and .json loaded on startup!")
        except Exception as e:
            print(f"⚠️ Error loading folder on startup: {e}")
            print("The app will start without pre-loaded data. Use /load/uniapnea to load data manually.")
    else:
        print(f"⚠️ Folder path does not exist: {folder_path}")
        
    app.run(debug=False)  # Set to False to avoid restart loop in production

Loading vectorstore on startup...
Processing PDF files...
Total PDF text length: 2683152 characters
Created 6831 PDF chunks
Processing JSON chunks...
Loaded 10 JSON chunks
Total chunks to process: 6841
Processing batch 1/69 (1.4%)
Processing batch 25/69 (36.2%)


Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x000001DCEED609A0>
Traceback (most recent call last):
  File "C:\Users\maryamata\AppData\Local\anaconda3\Lib\_weakrefset.py", line 40, in _remove
    self = selfref()
           ^^^^^^^^^
KeyboardInterrupt: 


KeyboardInterrupt: 